# Proyecto #2 - Data Mining
## Semana 4: Interpretabilidad del Modelo Final, Analisis de Errores y Cierre

Este notebook completa los requisitos de Semana 4 sobre el modelo final definido en Semana 3:
- Modelo final: **SVM optimizado (RBF) en Escenario A**.
- Enfoque 1: interpretabilidad tecnicamente defendible del modelo final.
- Enfoque 2: analisis de errores en test (con foco en FP, dado FN=0).
- Enfoque 3: evidencia reusable para informe y presentacion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42
TEST_SIZE = 0.20

print('Librerias cargadas correctamente.')

In [ ]:
# Rutas robustas
base_candidates = [Path('.'), Path('..')]
data_path = None
for b in base_candidates:
    p = b / 'data' / 'Datos_modelado.csv'
    if p.exists():
        data_path = p
        break
if data_path is None:
    raise FileNotFoundError('No se encontro data/Datos_modelado.csv')

week3_dir = data_path.parent / 'resultados_semana3_csv'
if not week3_dir.exists():
    raise FileNotFoundError('No se encontro data/resultados_semana3_csv')

week4_dir = data_path.parent / 'resultados_semana4_csv'
week4_dir.mkdir(parents=True, exist_ok=True)

print(f'Dataset: {data_path.resolve()}')
print(f'CSV Semana 3: {week3_dir.resolve()}')
print(f'Salida Semana 4: {week4_dir.resolve()}')

## 1) Carga de datos y reconstruccion del split de Semana 3

Se usa exactamente el mismo split estratificado 80/20 con random_state=42 para mantener comparabilidad directa con los resultados oficiales de Semana 3.

In [ ]:
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
target_col = 'Class'
id_col = 'Sample code number'
feature_cols = [c for c in df.columns if c not in [target_col, id_col]]

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Dataset total: {df.shape}')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print('Distribucion y_test:')
print(y_test.value_counts().sort_index())

## 2) Reentrenamiento del modelo final y comparador fuerte

- Modelo final (Semana 3): SVM optimizado en Escenario A.
- Comparador fuerte: Regresion Logistica optimizada en Escenario A.

In [ ]:
svm_final = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(C=0.1, class_weight=None, gamma=1, kernel='rbf', probability=True))
])

lr_opt = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000, penalty='l1', solver='liblinear'))
])

svm_final.fit(X_train, y_train)
lr_opt.fit(X_train, y_train)

y_pred_svm = svm_final.predict(X_test)
y_pred_lr = lr_opt.predict(X_test)

y_proba_svm = svm_final.predict_proba(X_test)[:, 1]
y_proba_lr = lr_opt.predict_proba(X_test)[:, 1]

print('Modelos entrenados y predicciones generadas.')

In [ ]:
def resumen_modelo(nombre, y_true, y_pred, y_score):
    cm = confusion_matrix(y_true, y_pred, labels=[2, 4])
    tn, fp, fn, tp = cm.ravel()
    return {
        'Modelo': nombre,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision_maligna': round(precision_score(y_true, y_pred, pos_label=4, zero_division=0), 4),
        'Recall_maligna': round(recall_score(y_true, y_pred, pos_label=4, zero_division=0), 4),
        'F1_maligna': round(f1_score(y_true, y_pred, pos_label=4, zero_division=0), 4),
        'AUC': round(roc_auc_score(y_true, y_score, labels=[2, 4]), 4),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }

resumen = pd.DataFrame([
    resumen_modelo('SVM_A_Optimizado', y_test, y_pred_svm, y_proba_svm),
    resumen_modelo('LR_A_Optimizada', y_test, y_pred_lr, y_proba_lr)
])

display(resumen)

print('Reporte de clasificacion SVM final:')
print(classification_report(y_test, y_pred_svm, digits=4))

In [ ]:
# Matriz de confusion visual del modelo final
cm_svm = confusion_matrix(y_test, y_pred_svm, labels=[2, 4])
cm_df = pd.DataFrame(cm_svm, index=['Real 2 (Benigno)', 'Real 4 (Maligno)'], columns=['Pred 2', 'Pred 4'])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title('SVM optimizado A - Matriz de confusion')
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
plt.tight_layout()
fig.savefig(week4_dir / '04_confusion_matrix_svm.png', dpi=200, bbox_inches='tight')
plt.show()

## 3) Analisis de errores del modelo final (foco FP)

Se identifican casos mal clasificados, su tipo de error y su comparacion frente a casos correctamente clasificados.

In [ ]:
errores = X_test.copy()
errores['y_real'] = y_test.values
errores['y_pred_svm'] = y_pred_svm
errores['y_pred_lr'] = y_pred_lr
errores['indice_original'] = errores.index

errores['tipo_error_svm'] = np.where(
    (errores['y_real'] == 4) & (errores['y_pred_svm'] == 2), 'FN',
    np.where((errores['y_real'] == 2) & (errores['y_pred_svm'] == 4), 'FP', 'OK')
)

errores['tipo_error_lr'] = np.where(
    (errores['y_real'] == 4) & (errores['y_pred_lr'] == 2), 'FN',
    np.where((errores['y_real'] == 2) & (errores['y_pred_lr'] == 4), 'FP', 'OK')
)

tabla_errores_svm = errores[errores['tipo_error_svm'] != 'OK'].copy()

print('Conteo de errores SVM:')
print(tabla_errores_svm['tipo_error_svm'].value_counts(dropna=False))
display(tabla_errores_svm[['indice_original'] + feature_cols + ['y_real', 'y_pred_svm', 'tipo_error_svm']])

In [ ]:
# Patrones en FP vs benignos correctamente clasificados
fp_svm = errores[errores['tipo_error_svm'] == 'FP'].copy()
tn_svm = errores[(errores['y_real'] == 2) & (errores['y_pred_svm'] == 2)].copy()

if len(fp_svm) > 0 and len(tn_svm) > 0:
    mean_fp = fp_svm[feature_cols].mean()
    mean_tn = tn_svm[feature_cols].mean()
    delta_fp_vs_tn = (mean_fp - mean_tn).sort_values(ascending=False)

    patrones_fp = pd.DataFrame({
        'Media_FP': mean_fp,
        'Media_TN_correcto': mean_tn,
        'Delta_FP_vs_TN': delta_fp_vs_tn
    }).sort_values('Delta_FP_vs_TN', ascending=False)

    display(patrones_fp.head(10))

    fig, ax = plt.subplots(figsize=(8, 5))
    top_delta = patrones_fp.head(8).iloc[::-1]
    ax.barh(top_delta.index, top_delta['Delta_FP_vs_TN'], color='#b33c4a', alpha=0.9)
    ax.set_title('Variables donde FP de SVM superan a TN correctos')
    ax.set_xlabel('Delta de media (FP - TN correcto)')
    plt.tight_layout()
    fig.savefig(week4_dir / '05_fp_vs_tn_delta.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('No hay suficientes casos para comparacion FP vs TN.')

## 4) Interpretabilidad complementaria (RF + LR)

Como SVM-RBF no entrega importancias directas, se construye una explicacion complementaria para el informe y la presentacion: 
- Random Forest para importancia por impureza.
- Regresion Logistica para magnitud de coeficientes estandarizados.
- Ranking promedio para robustez narrativa.

In [ ]:
rf_exp = RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced')
rf_exp.fit(X_train, y_train)
rf_importance = pd.Series(rf_exp.feature_importances_, index=feature_cols, name='Importancia_RF')

lr_exp = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000, penalty='l2', solver='lbfgs'))
])
lr_exp.fit(X_train, y_train)
lr_abscoef = pd.Series(np.abs(lr_exp.named_steps['model'].coef_[0]), index=feature_cols, name='Importancia_LR_abscoef')

tabla_interp = pd.concat([rf_importance, lr_abscoef], axis=1).reset_index().rename(columns={'index': 'Variable'})
tabla_interp['Rank_RF'] = tabla_interp['Importancia_RF'].rank(ascending=False, method='min')
tabla_interp['Rank_LR'] = tabla_interp['Importancia_LR_abscoef'].rank(ascending=False, method='min')
tabla_interp['Rank_Promedio'] = ((tabla_interp['Rank_RF'] + tabla_interp['Rank_LR']) / 2).round(1)
tabla_interp = tabla_interp.sort_values('Rank_Promedio').reset_index(drop=True)
display(tabla_interp)

In [ ]:
top_n = 8
plot_df = tabla_interp.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df['Variable'], plot_df['Importancia_RF'], color='#2f6db3', alpha=0.9, label='RF')
ax.set_title('Top variables por importancia (Random Forest explicativo)')
ax.set_xlabel('Importancia RF')
plt.tight_layout()
fig.savefig(week4_dir / '06_top_variables_rf.png', dpi=200, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df['Variable'], plot_df['Importancia_LR_abscoef'], color='#55a868', alpha=0.9, label='LR abs coef')
ax.set_title('Top variables por magnitud de coeficiente (LR explicativa)')
ax.set_xlabel('Magnitud |coef|')
plt.tight_layout()
fig.savefig(week4_dir / '07_top_variables_lr.png', dpi=200, bbox_inches='tight')
plt.show()

## 5) Exportacion de evidencia para informe y presentacion

Se guardan tablas reutilizables para documento final y exposicion.

In [ ]:
resumen.to_csv(week4_dir / '00_resumen_svm_vs_lr_test.csv', index=False, encoding='utf-8-sig')
tabla_errores_svm.to_csv(week4_dir / '01_svm_test_errors.csv', index=False, encoding='utf-8-sig')
errores.to_csv(week4_dir / '02_test_predictions_compare_svm_lr.csv', index=False, encoding='utf-8-sig')
tabla_interp.to_csv(week4_dir / '03_feature_rank_rf_lr.csv', index=False, encoding='utf-8-sig')

manifest = pd.DataFrame([
    ['00_resumen_svm_vs_lr_test.csv', 'Resumen comparativo de metricas y matriz de confusion'],
    ['01_svm_test_errors.csv', 'Casos mal clasificados por SVM final'],
    ['02_test_predictions_compare_svm_lr.csv', 'Predicciones comparadas SVM vs LR en test'],
    ['03_feature_rank_rf_lr.csv', 'Ranking de variables para interpretabilidad'],
    ['04_confusion_matrix_svm.png', 'Figura matriz de confusion de SVM final'],
    ['05_fp_vs_tn_delta.png', 'Figura de deltas FP vs TN correctos'],
    ['06_top_variables_rf.png', 'Figura top variables RF'],
    ['07_top_variables_lr.png', 'Figura top variables LR']
], columns=['archivo', 'descripcion'])
manifest.to_csv(week4_dir / '00_manifest_semana4.csv', index=False, encoding='utf-8-sig')

print(f'Evidencia exportada en: {week4_dir.resolve()}')
display(manifest)

## 6) Texto base para informe (Resultados y discusion)

### Interpretabilidad
La interpretabilidad del modelo final SVM-RBF se realizo mediante una estrategia complementaria con Random Forest y Regresion Logistica explicativos. El ranking combinado confirma como variables dominantes Bare Nuclei, Uniformity of Cell Shape y Uniformity of Cell Size, en coherencia con los hallazgos del EDA y la seleccion de variables de semanas previas.

### Analisis de errores
En test, el SVM final produjo TN=83, FP=5, FN=0 y TP=47. No hubo falsos negativos, por lo que el analisis se enfoco en falsos positivos. Los FP presentan rasgos mas altos en variables de atipia respecto a benignos correctamente clasificados, sugiriendo proximidad a la frontera de decision del modelo.

### Trade-off frente a LR optimizada
SVM evita 2 falsos negativos adicionales frente a LR optimizada (FN=0 vs FN=2), a cambio de 2 falsos positivos mas (FP=5 vs FP=3). Bajo el criterio clinico del proyecto, esta compensacion se considera aceptable por el mayor costo potencial de omitir casos malignos.

### Recomendacion
Mantener SVM optimizado en Escenario A como modelo final base, acompanado de monitoreo de FP y protocolo de segunda revision para casos positivos de baja confianza.